# M3 Pseudo-Labelling on IEEE-CIS

## 1. Introduction
M2 CatBoost can only train on labels that are at least 7 days old, so the newest 7 days before each retrain are not used for training, even though we can already see their features. In notebook 05 this delay cost M2 CatBoost 0.039 PR-AUC (0.653 with the 7-day delay vs 0.692 with the 0-day delay)

M3 tries to recover part of this using **confidence-based pseudo-labelling**, a form of self-training:
- M2 scores the transactions of the hidden week

- the transactions it is most confident about get a pseudo-label (the model's own guess, used as if it were a label)

- the model is retrained on the verified labels plus these pseudo-labels, with a lower weight on the pseudo-labels

M3 is built on the final M2 CatBoost from notebook 06 (115 selected features and tuned settings, validation PR-AUC 0.667), and these stay fixed so that any change comes from the pseudo-labels

The notebook covers:
- **Method** (Section 3): how each block is trained, and the weights of each type of row

- **Tuning blocks** (Section 4): we tune the thresholds and reliability weights together with Optuna on the training days, so the validation rows are not used to choose

- **Validation** (Section 5): we compare M3 with M2 on the validation rows, with the 0-day delay as a reference for how much there is to recover, and repeat M2 and M3 with 3 seeds

We only use the training and validation rows here. The test rows are kept aside for the final evaluation

In [25]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import optuna

from catboost import CatBoostClassifier
from sklearn.metrics import average_precision_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score

pd.set_option("display.max_columns", 100)
optuna.logging.set_verbosity(optuna.logging.WARNING)   # we print our own line per trial

SEED = 100
PROCESSED_DIR = Path("../data/processed")

# the selected features and the M2 scores from notebook 06
TUNING_DIR = Path("../data/interim/m2_tuning")

# everything this notebook trains is saved here, so a rerun loads the finished runs instead of training them again
RUNS_DIR = Path("../data/interim/m3_pseudo")
RUNS_DIR.mkdir(parents=True, exist_ok=True)

/Users/lycia/venv-notebook/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Data Loading and Preparation
We load the features the same way as notebook 06 (Section 2). The training and validation rows are kept in one frame (`X`, `y`) with the day of each row

In [26]:
features = pd.read_pickle(PROCESSED_DIR / "ieee_features.pkl")
with open(PROCESSED_DIR / "feature_lists.json") as f:
    feature_lists = json.load(f)

# these columns are stored as numbers, but the numbers are codes with no order
CODE_AS_CATEGORY = ["card1", "card2", "card3", "card5", "addr1", "addr2"]

feature_cols = feature_lists["ieee_feature_cols"]
cat_cols = []
num_cols = []
for col in feature_cols:
    if col in feature_lists["ieee_categorical_cols"] or col in CODE_AS_CATEGORY:
        cat_cols.append(col)
    else:
        num_cols.append(col)
v_drop = feature_lists["ieee_v_reduced_drop"]
cd_drop = feature_lists["ieee_cd_corr_drop"]

# keep only the needed columns, and only the training and validation rows
keep = feature_cols + ["isFraud", "ieee_split", "TransactionDT"]
features = features.loc[features["ieee_split"].isin(["train", "validation"]), keep]
features[num_cols] = features[num_cols].astype("float32")
for col in cat_cols:
    if col in CODE_AS_CATEGORY:
        # numbers to text codes e.g. 315.0 becomes "315" and a missing value becomes "missing"
        numbers = pd.to_numeric(features[col], errors="coerce")
        values = numbers.to_numpy(dtype="float64")
        rounded = np.round(values)
        whole = np.isfinite(values) & np.isclose(values, rounded)
        text = np.full(len(features), "missing", dtype=object)
        text[whole] = rounded[whole].astype("int64").astype(str)
        other = np.isfinite(values) & ~whole   # not a whole number => keep the value
        text[other] = numbers.to_numpy(dtype=object)[other]
        features[col] = pd.Series(text, index=features.index).astype("category")
    else:
        features[col] = features[col].astype("object").fillna("missing").astype(str).astype("category")

X, y = features[feature_cols], features["isFraud"]
day = (features["TransactionDT"] // 86400).to_numpy()
is_train = (features["ieee_split"] == "train").to_numpy()
is_val = ~is_train
del features

print("training rows:", is_train.sum(), ", days", day[is_train].min(), "to", day[is_train].max(), "| fraud rate:", round(y[is_train].mean() * 100, 2), "%")
print("validation rows:", is_val.sum(), ", days", day[is_val].min(), "to", day[is_val].max(), "| fraud rate:", round(y[is_val].mean() * 100, 2), "%")
print("features:", len(feature_cols), "\ncategorical (with the 6 code columns):", len(cat_cols), "\nV drop list:", len(v_drop), "\nC / D drop list:", len(cd_drop))

training rows: 377945 , days 1 to 109 | fraud rate: 3.41 %
validation rows: 94487 , days 109 to 141 | fraud rate: 3.92 %
features: 575 
categorical (with the 6 code columns): 47 
V drop list: 221 
C / D drop list: 6


### 2.1 Encodings
Same as notebook 06 (Section 2.1): the frequency features and the `card1` statistics are refitted before every retrain on all rows before the block start. They do not use the labels, so the hidden week can be used for them

In [27]:
# the frequency and card1 statistic features from notebook 04, which are fitted there once on all training rows
FREQ_SOURCE_COLS = []
for col in feature_cols:
    if col.endswith("_freq"):
        FREQ_SOURCE_COLS.append(col.removesuffix("_freq"))
CARD1_STAT_SOURCE_COLS = ["TransactionAmt", "D1", "D10", "D15"]

# count of each value and the card1 statistics, learned from encode_rows only (no labels are used)
def fit_encodings(encode_rows):
    X_enc = X.loc[encode_rows, FREQ_SOURCE_COLS + CARD1_STAT_SOURCE_COLS]
    counts = {}
    for col in FREQ_SOURCE_COLS:
        counts[col] = X_enc[col].value_counts()

    # float64 so the averages are the same as in notebook 04
    stats = {}
    for col in CARD1_STAT_SOURCE_COLS:
        stats[col] = X_enc[col].astype("float64").groupby(X_enc["card1"], observed=True).agg(["mean", "std"])
    return counts, stats

# replace the encoded columns of X_part (a value not in encode_rows gets a count of 0 and no statistics, as in notebook 04)
def apply_encodings(X_part, encodings):
    counts, stats = encodings
    for col in FREQ_SOURCE_COLS:
        X_part[f"{col}_freq"] = X_part[col].map(counts[col]).astype("float32").fillna(0)
    for col in CARD1_STAT_SOURCE_COLS:
        for stat in ["mean", "std"]:
            X_part[f"{col}_{stat}_by_card1"] = X_part["card1"].map(stats[col][stat]).astype("float32")
    return X_part

### 2.2 Selected Features and Settings
We use the final M2 CatBoost from notebook 06: the features with a PR-AUC drop above 0.0003 (Section 4.4) and the settings of the best Optuna trial (Section 5)

In [28]:
LABEL_DELAY_DAYS = 7
FREQUENCY = 7

# same as notebook 06, Section 4.4
MIN_DROP = 0.0003
permutation = pd.read_pickle(TUNING_DIR / "permutation_importance_348.pkl")

selected_cols = []
for col in feature_cols:
    if col in permutation.index and permutation.loc[col, "PR-AUC drop"] > MIN_DROP:
        selected_cols.append(col)

cat_features = []
for col in cat_cols:
    if col in selected_cols:
        cat_features.append(col)

# best trial of notebook 06, Section 5
TUNED_SETTINGS = {"learning_rate": 0.1, "iterations": 750, "depth": 8, "l2_leaf_reg": 1.5552439007420584, "random_strength": 0.1325358735488683,
                  "bagging_temperature": 0.42109136073074666, "rsm": 0.2810045903691265}

print("selected features:", len(selected_cols))
print("categorical:", len(cat_features))

selected features: 115
categorical: 21


## 3. Pseudo-Labelling
For a block that starts on day `d`:
1. **First model (M2)**: train on the labelled rows before day `d - 7` and score the hidden week (days `d - 7` to `d - 1`)

2. **Pseudo-labels**: the highest scores of the hidden week become pseudo-labelled fraud, and the lowest scores become pseudo-labelled legitimate. The rows in between are left out, since the model is not confident about them

3. **Second model (M3)**: train on the labelled rows plus the pseudo-labelled rows

4. **Scoring**: the second model scores days `d` to `d + 6`

The true labels of the hidden week are never used for training. We only use them in Section 4.2 to check how often the pseudo-labels are right, which would not be possible when the model is used in a real setting

**Settings we tune**

| Setting | What it means | Larger value means |
|---|---|---|
| `fraud_thres` | Share of the hidden week labelled as fraud, as a multiple of the fraud rate in the verified labels (e.g. 0.5 labels half as many rows as the fraud rate) | More pseudo-labelled fraud **BUT** less sure |
| `legit_thres` | Share of the hidden week with the lowest scores labelled as legitimate (e.g. 0.5 labels the lowest 50%) | More pseudo-labelled legitimate rows **BUT** less sure |
| `fraud_reliability` | How much a pseudo-labelled fraud row counts in training, where a verified row counts as 1 | We trust these rows more |
| `legit_reliability` | How much a pseudo-labelled legitimate row counts in training, where a verified row counts as 1 | We trust these rows more |

We use separate settings for fraud and legitimate rows since fraud is rare, so one threshold would give many confident legitimate rows and very few confident fraud rows.

**Settings we do not tune**: the class weights, CatBoost settings, features and retraining stay the same as M2, so the only change from M2 is the pseudo-labels. M2 CatBoost has no class weights, since they lowered PR-AUC in M1 (notebook 05, Section 5) and in the tuning search in notebook 06

| Row type | Weight in training |
|---|---|
| Verified legitimate | 1 |
| Verified fraud | 1 |
| Pseudo-labelled legitimate | `legit_reliability` |
| Pseudo-labelled fraud | `fraud_reliability` |

Note: since we are using the same M2 model forM3, the risk of this method is that the model learns from its own mistakes => this is why we only use the confident rows and give them a lower weight, and why Section 4.2 checks the pseudo-labels

In [29]:
# M2 for one block: trains on the labelled rows, scores the hidden week and the block's days, saved so a rerun loads it
# allowed_rows are the rows we may train on, and split_rows are the rows we are evaluating
def first_model(run_name, allowed_rows, split_rows, block_start, seed, delay=LABEL_DELAY_DAYS):
    path = RUNS_DIR / f"first_{run_name}_seed{seed}_block{block_start}.pkl"
    if path.exists():
        return pd.read_pickle(path)

    start = time.time()
    fit_rows = allowed_rows & (day < block_start - delay)
    hidden_rows = allowed_rows & (day >= block_start - delay) & (day < block_start)
    score_rows = split_rows & (day >= block_start) & (day < block_start + FREQUENCY)
    encodings = fit_encodings(allowed_rows & (day < block_start))

    X_fit = apply_encodings(X[fit_rows], encodings)[selected_cols]
    X_score = apply_encodings(X[score_rows], encodings)[selected_cols]
    model = CatBoostClassifier(random_seed=seed, thread_count=-1, verbose=0, allow_writing_files=False, **TUNED_SETTINGS)
    model.fit(X_fit, y[fit_rows], cat_features=cat_features)

    # with the 0-day delay there is no hidden week
    hidden_scores = pd.Series(dtype="float64")
    if hidden_rows.sum() > 0:
        X_hidden = apply_encodings(X[hidden_rows], encodings)[selected_cols]
        hidden_scores = pd.Series(model.predict_proba(X_hidden)[:, 1], index=X.index[hidden_rows])
    block_scores = pd.Series(model.predict_proba(X_score)[:, 1], index=X.index[score_rows])

    result = {"hidden": hidden_scores, "block": block_scores}
    pd.to_pickle(result, path)
    print(f"{run_name}, block {block_start}, seed {seed}: first model ({time.time() - start:.0f}s)")
    return result


# all blocks of the first model, as one series of scores
def m2_run(run_name, allowed_rows, split_rows, block_starts, seed, delay=LABEL_DELAY_DAYS):
    block_scores = []
    for block_start in block_starts:
        block_scores.append(first_model(run_name, allowed_rows, split_rows, block_start, seed, delay)["block"])
    return pd.concat(block_scores)


# class-specific thresholds: returns which rows of the hidden week are pseudo-labelled fraud and legitimate
def pseudo_labels(hidden_scores, fraud_rate, config):
    sorted_scores = np.sort(hidden_scores)
    n_fraud = max(int(round(config["fraud_thres"] * fraud_rate * len(sorted_scores))), 1)
    n_legit = int(round(config["legit_thres"] * len(sorted_scores)))
    fraud_threshold = sorted_scores[-n_fraud]
    legit_threshold = sorted_scores[n_legit - 1]
    return hidden_scores >= fraud_threshold, hidden_scores <= legit_threshold


# M3 for all blocks: the second model trains on the labelled rows plus the pseudo-labelled rows, saved so a rerun loads it
def m3_run(run_name, allowed_rows, split_rows, block_starts, config, seed, trial=None):
    config_name = f"{config['fraud_thres']}_{config['legit_thres']}_{config['fraud_reliability']}_{config['legit_reliability']}"
    path = RUNS_DIR / f"m3_{run_name}_{config_name}_seed{seed}.pkl"
    if path.exists():
        return pd.read_pickle(path)

    start = time.time()
    all_scores = []
    for i, block_start in enumerate(block_starts):
        first = first_model(run_name, allowed_rows, split_rows, block_start, seed)
        fit_rows = allowed_rows & (day < block_start - LABEL_DELAY_DAYS)
        hidden_rows = allowed_rows & (day >= block_start - LABEL_DELAY_DAYS) & (day < block_start)
        score_rows = split_rows & (day >= block_start) & (day < block_start + FREQUENCY)

        # only verified labels are used for the fraud rate
        hidden_scores = first["hidden"].to_numpy()
        is_fraud, is_legit = pseudo_labels(hidden_scores, y[fit_rows].mean(), config)
        keep = is_fraud | is_legit

        encodings = fit_encodings(allowed_rows & (day < block_start))
        X_labelled = apply_encodings(X[fit_rows], encodings)[selected_cols]
        X_hidden = apply_encodings(X[hidden_rows], encodings)[selected_cols]
        X_score = apply_encodings(X[score_rows], encodings)[selected_cols]

        # pseudo-labels and their weights
        hidden_labels = is_fraud[keep].astype(int)
        hidden_weights = np.where(is_fraud[keep], config["fraud_reliability"], config["legit_reliability"])

        X_fit = pd.concat([X_labelled, X_hidden[keep]])
        y_fit = np.concatenate([y[fit_rows].to_numpy(), hidden_labels])
        weights = np.concatenate([np.ones(len(X_labelled)), hidden_weights])

        model = CatBoostClassifier(random_seed=seed, thread_count=-1, verbose=0, allow_writing_files=False, **TUNED_SETTINGS)
        model.fit(X_fit, y_fit, cat_features=cat_features, sample_weight=weights)
        all_scores.append(pd.Series(model.predict_proba(X_score)[:, 1], index=X.index[score_rows]))
        del model, X_fit, X_labelled, X_hidden

        # Optuna: report the pooled PR-AUC of the blocks so far, and stop the trial early if it is much worse than the earlier trials
        if trial is not None:
            scores_so_far = pd.concat(all_scores)
            trial.report(average_precision_score(y.loc[scores_so_far.index], scores_so_far), i)
            if trial.should_prune():
                raise optuna.TrialPruned()

    scores = pd.concat(all_scores)
    scores.to_pickle(path)
    print(f"{run_name}, {config_name}, seed {seed}: second models ({time.time() - start:.0f}s)")
    return scores

## 4. Tuning Blocks
We use the same 5 tuning blocks as notebook 06 (Section 3), which start on days 81, 88, 95, 102 and 109. They only use the training rows, and every choice uses the pooled PR-AUC

### 4.1 M2 on the Tuning Blocks
The first models are M2, so their pooled PR-AUC should give the best trial of notebook 06 (0.5868) => this checks that the setup is the same as notebook 06

In [30]:
TUNING_BLOCKS = [81, 88, 95, 102, 109]

tuning_m2_scores = m2_run("tuning", is_train, is_train, TUNING_BLOCKS, SEED)
tuning_m2 = average_precision_score(y.loc[tuning_m2_scores.index], tuning_m2_scores)
print(f"M2 on the tuning blocks: {tuning_m2:.4f} (notebook 06, Section 5: 0.5868)")

M2 on the tuning blocks: 0.5868 (notebook 06, Section 5: 0.5868)


### 4.2 Tuning with Optuna
We tune these 4 settings together, since they all control how much the pseudo-labels affect the model, so the best threshold can depend on the reliability weights:
- `fraud_thres`: 0.1 to 1.5 (a multiple of the fraud rate, so it can be above 1 e.g. with 3.4% fraud, 1.5 labels the top 5% of the hidden week)

- `legit_thres`: 0.1 to 0.95 (a share of the hidden week, so it stops below 1, since 1 would label every row as legitimate)

- `fraud_reliability`: 0.1 to 1 (1 = pseudo-labelled row counts exactly as much in training as a verified row)

- `legit_reliability`: 0.1 to 1 (1 = pseudo-labelled row counts exactly as much in training as a verified row)

How the search runs (same as notebook 06, Section 5):
- **First trial**: `fraud_thres` 0.5, `legit_thres` 0.5 and both reliability weights at 0.3, as a starting point

- **Sampler**: Optuna's default TPE sampler, which picks the next settings close to the settings that scored well so far

- **Pruning**: after each block, Optuna compares the pooled PR-AUC of the blocks so far with the other trials after the same block, and stops the trial if it is below their median (only after the first 5 trials)

- **Saving**: the trials are saved in `data/interim/m3_pseudo/optuna.db`

Each trial only trains the second models (M3), since the first models (M2) from Section 4.1 are saved

In [ ]:
N_TRIALS = 40


def objective(trial):
    config = {
        "fraud_thres": trial.suggest_float("fraud_thres", 0.1, 1.5, log=True),
        "legit_thres": trial.suggest_float("legit_thres", 0.1, 0.95),
        "fraud_reliability": trial.suggest_float("fraud_reliability", 0.1, 1, log=True),
        "legit_reliability": trial.suggest_float("legit_reliability", 0.1, 1, log=True)
    }

    start = time.time()
    try:
        scores = m3_run("tuning", is_train, is_train, TUNING_BLOCKS, config, SEED, trial=trial)
    except optuna.TrialPruned:
        print(f"trial {trial.number}: pruned ({time.time() - start:.0f}s)")
        raise
    pooled = average_precision_score(y.loc[scores.index], scores)
    print(f"trial {trial.number}: PR-AUC {pooled:.4f} ({time.time() - start:.0f}s)")
    return float(pooled)


study = optuna.create_study(study_name="m3_pseudo_labelling", storage=f"sqlite:///{RUNS_DIR / 'optuna.db'}",
                            direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED),
                            pruner=optuna.pruners.MedianPruner(n_startup_trials=5), load_if_exists=True)

# only run the trials that are still missing (a trial that was stopped halfway is not counted)
finished = 0
for t in study.trials:
    if t.state == optuna.trial.TrialState.COMPLETE or t.state == optuna.trial.TrialState.PRUNED:
        finished += 1

# run the starting point first
if finished == 0:
    study.enqueue_trial({"fraud_thres": 0.5, "legit_thres": 0.5, "fraud_reliability": 0.3, "legit_reliability": 0.3})

trials_left = max(N_TRIALS - finished, 0)
start = time.time()
study.optimize(objective, n_trials=trials_left)
print(f"{len(study.trials)} trials in the study ({time.time() - start:.0f}s in this run)")

trial 0: PR-AUC 0.5803 (0s)
tuning, 0.43560594895413396_0.33661397732972675_0.2657771198261658_0.6994813383440696, seed 100: second models (1715s)
trial 1: PR-AUC 0.5818 (1715s)
tuning, 0.10128608985061957_0.2033337526656471_0.4685426016771214_0.6696575276234319, seed 100: second models (1616s)
trial 2: PR-AUC 0.5767 (1616s)
tuning, 0.14480362203265254_0.5888293300131625_0.7786135445417824_0.16188332728174953, seed 100: second models (1644s)
trial 3: PR-AUC 0.5867 (1644s)
tuning, 0.1651820680381374_0.19212035689461687_0.16584313263890038_0.9519711448539989, seed 100: second models (1674s)
trial 4: PR-AUC 0.5767 (1674s)
tuning, 0.9007690523993594_0.24614986082270507_0.6549750379683221_0.187963596850642, seed 100: second models (1728s)
trial 5: PR-AUC 0.5303 (1729s)
trial 6: pruned (324s)
trial 7: pruned (321s)
trial 8: pruned (321s)
trial 9: pruned (319s)
trial 10: pruned (329s)
trial 11: pruned (325s)
tuning, 0.3289935276784793_0.5587476390294129_0.7246891019150818_0.1249405352286524, 

In [32]:
trials = study.trials_dataframe(attrs=("number", "value", "params", "state"))
new_columns = []
for col in trials.columns:
    new_columns.append(col.removeprefix("params_"))
trials.columns = new_columns
trials = trials.rename(columns={"value": "PR-AUC"})

complete = trials[trials["state"] == "COMPLETE"].sort_values("PR-AUC", ascending=False)
complete["change from M2"] = complete["PR-AUC"] - tuning_m2
print("trials:", len(trials), "| complete:", len(complete), "| pruned:", (trials["state"] == "PRUNED").sum(), "| above M2:", (complete["change from M2"] > 0).sum())
print("best 10 trials:")
display(complete.drop(columns="state").head(10).round(4))

# how much each setting changes the score across the complete trials
print("importance of each setting (Optuna fANOVA):")
display(pd.Series(optuna.importance.get_param_importances(study)).rename("importance").round(3).to_frame())

best_config = study.best_trial.params
print("best settings:", best_config)
print(f"M2 {tuning_m2:.4f} | best M3 {study.best_value:.4f}")

trials: 40 | complete: 9 | pruned: 31 | above M2: 0
best 10 trials:


,number,PR-AUC,fraud_reliability,fraud_thres,legit_reliability,legit_thres,change from M2
3,3,0.5867,0.7786,0.1448,0.1619,0.5888,-0.0001
12,12,0.5852,0.7247,0.3290,0.1249,0.5587,-0.0017
13,13,0.5830,0.6797,0.2756,0.1109,0.5436,-0.0038
1,1,0.5818,0.2658,0.4356,0.6995,0.3366,-0.0050
0,0,0.5803,0.3000,0.5000,0.3000,0.5000,-0.0065
2,2,0.5767,0.4685,0.1013,0.6697,0.2033,-0.0101
4,4,0.5767,0.1658,0.1652,0.9520,0.1921,-0.0101
38,38,0.5651,0.5994,0.6167,0.1391,0.5804,-0.0217
5,5,0.5303,0.6550,0.9008,0.1880,0.2461,-0.0565


importance of each setting (Optuna fANOVA):


,importance
fraud_thres,0.299
legit_reliability,0.263
legit_thres,0.225
fraud_reliability,0.213


best settings: {'fraud_thres': 0.14480362203265254, 'legit_thres': 0.5888293300131625, 'fraud_reliability': 0.7786135445417824, 'legit_reliability': 0.16188332728174953}
M2 0.5868 | best M3 0.5867


In [38]:
# PR-AUC of each tuning trial in each block, with M2 as the first row (only loads saved scores)
# pruned trials only finished the first block, so the other blocks are empty for them
m2_row = {"trial": "M2", "state": ""}
for block_start in TUNING_BLOCKS:
    first = first_model("tuning", is_train, is_train, block_start, SEED)
    m2_row[block_start] = average_precision_score(y.loc[first["block"].index], first["block"])
m2_row["pooled"] = tuning_m2

weekly_rows = []
for t in study.trials:
    row = {"trial": t.number, "state": t.state.name}
    if t.state == optuna.trial.TrialState.COMPLETE:
        p = t.params
        config_name = f"{p['fraud_thres']}_{p['legit_thres']}_{p['fraud_reliability']}_{p['legit_reliability']}"
        scores = pd.read_pickle(RUNS_DIR / f"m3_tuning_{config_name}_seed{SEED}.pkl")
        for block_start in TUNING_BLOCKS:
            first = first_model("tuning", is_train, is_train, block_start, SEED)
            block_index = first["block"].index
            row[block_start] = average_precision_score(y.loc[block_index], scores.loc[block_index])
        row["pooled"] = t.value
    elif t.state == optuna.trial.TrialState.PRUNED:
        row[TUNING_BLOCKS[0]] = t.intermediate_values[0]   # the pooled score after 1 block is the score of that block
    weekly_rows.append(row)

weekly = pd.DataFrame(weekly_rows).set_index("trial")
weekly = weekly.sort_values("pooled", ascending=False)
weekly = pd.concat([pd.DataFrame([m2_row]).set_index("trial"), weekly])

print("PR-AUC in each block (columns: the day each block starts):")
display(weekly.round(4))

PR-AUC in each block (columns: the day each block starts):


,state,81,88,95,102,109,pooled
trial,,,,,,,
M2,,0.6090,0.5666,0.5154,0.6559,0.5309,0.5868
3,COMPLETE,0.6095,0.5614,0.5170,0.6570,0.5435,0.5867
12,COMPLETE,0.6015,0.5688,0.5175,0.6557,0.5120,0.5852
13,COMPLETE,0.6184,0.5570,0.5074,0.6507,0.4925,0.5830
1,COMPLETE,0.5986,0.5544,0.5196,0.6551,0.5381,0.5818
0,COMPLETE,0.5840,0.5712,0.5158,0.6494,0.5377,0.5803
2,COMPLETE,0.5712,0.5581,0.5218,0.6519,0.5292,0.5767
4,COMPLETE,0.5894,0.5580,0.5131,0.6459,0.5151,0.5767
38,COMPLETE,0.6111,0.5646,0.4982,0.6127,0.5297,0.5651


**Takeaway: Tuning**
- None of the 9 complete trials beat M2 on the tuning blocks (0.5868). The best is trial 3 at 0.5867 (-0.0001) => at best, M3 is level with M2 here

- Best settings (trial 3): `fraud_thres` 0.145, `legit_thres` 0.589, `fraud_reliability` 0.779 and `legit_reliability` 0.162

- Trial 3 is about that same as M2 in every block: +0.0005, -0.0052, +0.0016 and +0.0011 for the blocks starting on days 81, 88, 95 and 102, and +0.0126 for the block starting on day 109, which only has 996 rows => there is no block where the pseudo-labels clearly help

- The best 3 trials all pseudo-label only a little fraud with a high weight, and give the pseudo-labelled legitimate rows a low weight:
    - `fraud_thres` 0.14 to 0.33, `fraud_reliability` 0.68 to 0.78 and `legit_reliability` 0.11 to 0.16
    - pseudo-labelling more fraud makes it worse: the 2 trials with `fraud_thres` above 0.6 are the worst (0.5651 at 0.62 and 0.5303 at 0.90). Section 4.3 below shows that this is because many of these pseudo-labels are wrong

- The 4 settings have about the same Optuna importance (0.21 to 0.30) => no single setting drives the score **BUT** this comes from only 9 complete trials, so it is not very reliable

- There are 31 pruned trials that were stopped after the first block (day 81), and the first block does not predict the final score well e.g. trial 13 had the highest score after day 81 (0.6184) **BUT** ended 4th (0.5830), and trial 5 was 0.6087 after day 81 **BUT** ended as the worst trial (0.5303) => some pruned trials might have scored higher if they had finished

### 4.3 Pseudo-Label Quality
We check how often the pseudo-labels are right, using the true labels of the hidden week, for the best settings and for a range of thresholds

We choose the best trial only on the tuning blocks, even if it is not above M2, so that Section 5 is a fair test

Note: this score is the best of the 9 complete trials on the same blocks, so it looks better than it really is => the validation rows in Section 5 are the real check

In [ ]:
# how often the pseudo-labels are right, using the true labels of the hidden week
fraud_rows = []
for fraud_thres in [0.1, 0.25, 0.5, 1, 1.5, best_config["fraud_thres"]]:
    config = {"fraud_thres": fraud_thres, "legit_thres": 0.5}
    labelled = 0
    right = 0
    for block_start in TUNING_BLOCKS:
        first = first_model("tuning", is_train, is_train, block_start, SEED)
        fit_rows = is_train & (day < block_start - LABEL_DELAY_DAYS)
        true_labels = y.loc[first["hidden"].index].to_numpy()
        is_fraud, is_legit = pseudo_labels(first["hidden"].to_numpy(), y[fit_rows].mean(), config)
        labelled += is_fraud.sum()
        right += true_labels[is_fraud].sum()
    fraud_rows.append({"fraud_thres": fraud_thres, "pseudo-labelled fraud rows": labelled, "share really fraud": right / labelled})

legit_rows = []
for legit_thres in [0.1, 0.25, 0.5, 0.75, 0.95, best_config["legit_thres"]]:
    config = {"fraud_thres": 0.5, "legit_thres": legit_thres}
    labelled = 0
    right = 0
    for block_start in TUNING_BLOCKS:
        first = first_model("tuning", is_train, is_train, block_start, SEED)
        fit_rows = is_train & (day < block_start - LABEL_DELAY_DAYS)
        true_labels = y.loc[first["hidden"].index].to_numpy()
        is_fraud, is_legit = pseudo_labels(first["hidden"].to_numpy(), y[fit_rows].mean(), config)
        labelled += is_legit.sum()
        right += (true_labels[is_legit] == 0).sum()
    legit_rows.append({"legit_thres": legit_thres, "pseudo-labelled legitimate rows": labelled, "share really legitimate": right / labelled})

print("the last row of each table is the best trial")
display(pd.DataFrame(fraud_rows).round(4))
display(pd.DataFrame(legit_rows).round(4))

the last row of each table is the best trial


,fraud_thres,pseudo-labelled fraud rows,share really fraud
0,0.1000,378,0.9735
1,0.2500,943,0.9682
2,0.5000,1889,0.8925
3,1.0000,3778,0.6347
4,1.5000,5665,0.4688
5,0.1448,547,0.9744


,legit_thres,pseudo-labelled legitimate rows,share really legitimate
0,0.1000,11222,0.9975
1,0.2500,28054,0.9980
2,0.5000,56109,0.9968
3,0.7500,84165,0.9949
4,0.9500,106607,0.9867
5,0.5888,66077,0.9964


In [39]:
hidden_rows = 0
hidden_frauds = 0
for block_start in TUNING_BLOCKS:
    first = first_model("tuning", is_train, is_train, block_start, SEED)
    hidden_index = first["hidden"].index
    hidden_rows += len(hidden_index)
    hidden_frauds += y.loc[hidden_index].sum()
print("rows in the hidden weeks:", hidden_rows, "| frauds:", hidden_frauds, "| fraud rate:", round(hidden_frauds / hidden_rows, 4))

rows in the hidden weeks: 112219 | frauds: 4064 | fraud rate: 0.0362


**Takeaway: Pseudo-Label Quality**
- The fraud pseudo-labels are only reliable when very few rows are pseudo-labelled: about 97% are really fraud at `fraud_thres` 0.1 to 0.25, **BUT** this drops to 89% at 0.5, 63% at 1 and 47% at 1.5 => at 1.5 more than half of the pseudo-labelled fraud rows are legitimate, which explains why the trials with a high `fraud_thres` score the worst in Section 4.2

- The best trial pseudo-labels 547 fraud rows across the 5 hidden weeks and 97.4% of them (~533) are really fraud **BUT** the hidden weeks have a total of 4,064 frauds in total so this only makes up 13% of the frauds, and these are the frauds that M2 already scores the highest => they add little that the model does not already know

- Most legitimate pseudo-labels are right (98.7% to 99.8%) **BUT** there are so many of them that the few wrong ones still add up e.g. with the best trial, 0.36% of 66,077 rows is about 240 real frauds labelled as legitimate => this may be why the best trials give the legitimate pseudo-labels a low weight (0.11 to 0.16)

## 5. Validation Results
We rerun M2 and M3 on the validation rows the same way as notebook 06 (Section 6): retrained every 7 days on all earlier labelled days, with the encodings refitted before each block. We compare 3 versions:
1. **M2 with the 7-day delay**: the final model of notebook 06, which should give its validation PR-AUC (0.6673)

2. **M3 with the 7-day delay**: M2 plus the pseudo-labels, with the settings chosen in Section 4

3. **M2 with the 0-day delay**: all labels are known straight away, which is not realistic => this shows the most that M3 could recover

**Share of the gap recovered** = (M3 - M2 with the 7-day delay) / (M2 with the 0-day delay - M2 with the 7-day delay)

In [34]:
VAL_FIRST_DAY = int(day[is_val].min())
VAL_LAST_DAY = int(day[is_val].max())
VAL_BLOCKS = list(range(VAL_FIRST_DAY, VAL_LAST_DAY + 1, FREQUENCY))
labelled_rows = is_train | is_val

val_runs = {}
val_runs["M2, 7-day delay"] = m2_run("val", labelled_rows, is_val, VAL_BLOCKS, SEED)
val_runs["M3, 7-day delay"] = m3_run("val", labelled_rows, is_val, VAL_BLOCKS, best_config, SEED)
val_runs["M2, 0-day delay"] = m2_run("val_no_delay", labelled_rows, is_val, VAL_BLOCKS, SEED, delay=0)
m2_val_scores = val_runs["M2, 7-day delay"]
print(f"M2 with the 7-day delay: {average_precision_score(y.loc[m2_val_scores.index], m2_val_scores):.4f} (notebook 06, Section 6: 0.6673)")

val, 0.14480362203265254_0.5888293300131625_0.7786135445417824_0.16188332728174953, seed 100: second models (2296s)
M2 with the 7-day delay: 0.6673 (notebook 06, Section 6: 0.6673)


In [35]:
k = 100


def evaluate(name, series):
    scores = series.to_numpy()
    y_part = y.loc[series.index].to_numpy()
    day_part = pd.Series(day, index=X.index).loc[series.index].to_numpy()
    predictions = (scores >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_part, predictions, labels=[0, 1]).ravel()

    # precision@k per day: share of fraud among each day's k highest scores, averaged over the days
    daily = pd.DataFrame({"day": day_part, "y": y_part, "score": scores})
    top_k = daily.sort_values("score", ascending=False, kind="stable").groupby("day").head(k)

    return {"model": name,
            "PR-AUC": average_precision_score(y_part, scores),
            "ROC-AUC": roc_auc_score(y_part, scores),
            "precision": precision_score(y_part, predictions, zero_division=0),
            "recall": recall_score(y_part, predictions, zero_division=0),
            "F1": f1_score(y_part, predictions, zero_division=0),
            f"precision@{k} per day": top_k.groupby("day")["y"].mean().mean(),
            "TP": tp, "FP": fp, "FN": fn, "TN": tn}


result_rows = []
for name, series in val_runs.items():
    result_rows.append(evaluate(name, series))
results = pd.DataFrame(result_rows).set_index("model")
display(results.round(4))

m2_delay = results.loc["M2, 7-day delay", "PR-AUC"]
m3_delay = results.loc["M3, 7-day delay", "PR-AUC"]
m2_no_delay = results.loc["M2, 0-day delay", "PR-AUC"]
print(f"cost of the delay: {m2_no_delay - m2_delay:.4f} | M3 - M2: {m3_delay - m2_delay:+.4f} | share of the gap recovered: {(m3_delay - m2_delay) / (m2_no_delay - m2_delay):.1%}")

# PR-AUC per validation week
weekly = []
for name, series in val_runs.items():
    for i, block_start in enumerate(VAL_BLOCKS):
        block_day = pd.Series(day, index=X.index).loc[series.index].to_numpy()
        rows_w = (block_day >= block_start) & (block_day < block_start + FREQUENCY)
        weekly.append({"model": name, "week": i + 1, "PR-AUC": average_precision_score(y.loc[series.index].to_numpy()[rows_w], series.to_numpy()[rows_w])})
display(pd.DataFrame(weekly).pivot_table(index="model", columns="week", values="PR-AUC", sort=False).round(4))

,PR-AUC,ROC-AUC,precision,recall,F1,precision@100 per day,TP,FP,FN,TN
model,,,,,,,,,,
"M2, 7-day delay",0.6673,0.9374,0.8589,0.4619,0.6008,0.6467,1711,281,1993,90502
"M3, 7-day delay",0.6644,0.9367,0.8579,0.4627,0.6012,0.6370,1714,284,1990,90499
"M2, 0-day delay",0.7075,0.9426,0.8714,0.5124,0.6454,0.6800,1898,280,1806,90503


cost of the delay: 0.0402 | M3 - M2: -0.0029 | share of the gap recovered: -7.2%


week,1,2,3,4,5
model,,,,,
"M2, 7-day delay",0.6924,0.6394,0.6983,0.6805,0.6082
"M3, 7-day delay",0.6817,0.6443,0.7075,0.6564,0.6016
"M2, 0-day delay",0.7129,0.6780,0.7517,0.7278,0.6608


**Takeaway: Validation Results**
- With seed 100, M3 is slightly below M2: validation PR-AUC 0.6644 vs 0.6673 (-0.0029), so it recovers -7% of the 0.040 that the delay costs. ROC-AUC is about the same (0.9367 vs 0.9374)

- At the 0.5 threshold M3 and M2 are almost the same: 1,714 frauds caught and 284 false alerts vs 1,711 and 281. Precision@100 per day is a bit lower (0.637 vs 0.647)

- M3 does not win consistently across the weeks: it is higher in weeks 2 and 3 (+0.005 and +0.009) **BUT** lower in weeks 1, 4 and 5 (-0.011, -0.024 and -0.007)

- The 0-day delay is higher than M2 in all 5 weeks (+0.021 to +0.053), so the delay really does cost PR-AUC **BUT** the pseudo-labels do not recover it in any of the weeks => a likely reason is that the newest labels help most for the frauds the model is not sure about, and these are exactly the rows that pseudo-labelling leaves out (Section 4.3)

### 5.1 Seeds
The gain from M3 could be small compared with the seed noise (the PR-AUC range of M0 CatBoost over 3 seeds was 0.006, notebook 05 Section 5.6), and this range was not measured on M2. So we repeat M2 and M3 with 3 seeds, with the settings from Section 4 staying fixed

In [36]:
SEEDS = [100, 200, 300]

seed_rows = []
for seed in SEEDS:
    m2_scores = m2_run("val", labelled_rows, is_val, VAL_BLOCKS, seed)
    m3_scores = m3_run("val", labelled_rows, is_val, VAL_BLOCKS, best_config, seed)
    m2_score = average_precision_score(y.loc[m2_scores.index], m2_scores)
    m3_score = average_precision_score(y.loc[m3_scores.index], m3_scores)
    seed_rows.append({"seed": seed, "M2": m2_score, "M3": m3_score, "M3 - M2": m3_score - m2_score})

seed_table = pd.DataFrame(seed_rows).set_index("seed")
display(seed_table.round(4))
print(f"M2 mean {seed_table['M2'].mean():.4f}, range {seed_table['M2'].max() - seed_table['M2'].min():.4f}")
print(f"M3 mean {seed_table['M3'].mean():.4f}, range {seed_table['M3'].max() - seed_table['M3'].min():.4f}")
print(f"mean M3 - M2: {seed_table['M3 - M2'].mean():+.4f}")

val, 0.14480362203265254_0.5888293300131625_0.7786135445417824_0.16188332728174953, seed 200: second models (2375s)
val, 0.14480362203265254_0.5888293300131625_0.7786135445417824_0.16188332728174953, seed 300: second models (2317s)


,M2,M3,M3 - M2
seed,,,
100,0.6673,0.6644,-0.0029
200,0.6587,0.6667,0.0080
300,0.6599,0.6649,0.0049


M2 mean 0.6620, range 0.0086
M3 mean 0.6653, range 0.0023
mean M3 - M2: +0.0033


**Takeaway: Seeds**
- Over 3 seeds, M3 is a bit higher on average: mean PR-AUC 0.6653 vs 0.6620 (+0.0033). M3 is higher with seeds 200 and 300 (+0.008 and +0.005) BUT lower with seed 100 (-0.003), which happens to be M2’s best seed (0.6673 vs 0.6587 and 0.6599) and M3's worst seed (0.6644 vs 0.6667 and 0.6649)=> this is why M3 looked worse than M2 in Section 5
- **BUT** the mean gain (+0.0033) is smaller than the seed range of M2 (0.0086) => we cannot say that M3 is better than M2, and at most it recovers about 8% of the 0.040 that the delay costs. M3 does vary less across the 3 seeds (range 0.0023 vs 0.0086)
- The seed range of M2 (0.0086) is larger than the 0.006 of M0 CatBoost (notebook 05, Section 5.6) => the +0.014 gain from tuning in notebook 06 is still above 0.0086

- M3 also takes about twice as long to train, since each block trains 2 models

Pseudo-labelling does not clearly recover the cost of the label delay on IEEE-CIS => we keep M2 CatBoost as the main model